In [ ]:
import os
import time
from pathlib import Path
import jax
import jax.numpy as jp
import mujoco
from mujodico import mjx
import numpy as np
import cyipopt
import matplotlib.pyplot as plt
import viser
from mjviser import ViserMujocoScene

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
path = Path.cwd() / "1d-pusher.xml"
model = mujoco.MjModel.from_xml_path(str(path))

mjx_smooth_model = mjx.put_model(
    model,
    contact_approximation="log barrier",
    contact_smoothing=1e1,
)

mjx_model = mjx.put_model(
    model,
    contact_approximation="log barrier",
    contact_smoothing=0.0,
)

mjx_data = mjx.make_data(mjx_model)

nq = model.nq
nv = model.nv
nu = model.nu

@jax.jit
def step(x, u):
    d = mjx_data.replace(qpos=x[:nq], qvel=x[nq:], ctrl=u)
    d0 = mjx.step(mjx_model, d)
    xnext = jp.concatenate((d0.qpos, d0.qvel))
    d1 = mjx.step(mjx_smooth_model, d)
    xnext_smooth = jp.concatenate((d1.qpos, d1.qvel))
    return jax.lax.stop_gradient(xnext) + xnext_smooth - jax.lax.stop_gradient(xnext_smooth)

In [ ]:
N = 100

x0 = jp.array([0.0, 0.0, 0.0, 0.0])
x_goal = jp.array([0.0, 1.0, 0.0, 0.0])

Q = jp.diag(jp.array([0.0, 1.0, 0.0, 1.0]))
Qf = jp.diag(jp.array([0.0, 100.0, 0.0, 10.0]))
R = jp.eye(nu) * 0.10


nx = nq + nv
nz = (N + 1) * nx + N * nu

def unpack(z):
    xs = z[: (N + 1) * nx].reshape(N + 1, nx)
    us = z[(N + 1) * nx :].reshape(N, nu)
    return xs, us

@jax.jit
def objective(z):
    xs, us = unpack(z)

    cost = 0.0

    for k in range(N):
        dx = xs[k] - x_goal
        cost += dx @ Q @ dx
        cost += us[k] @ R @ us[k]

    dx = xs[-1] - x_goal
    cost += dx @ Qf @ dx

    return cost

objective_grad = jax.jit(jax.grad(objective))

@jax.jit
def constraints(z):
    xs, us = unpack(z)

    c = []

    for k in range(N):
        c.append(xs[k + 1] - step(xs[k], us[k]))

    return jp.concatenate(c)

constraint_jac = jax.jit(jax.jacfwd(constraints))

In [ ]:
class NLP:
    def objective(self, z):
        return float(objective(jp.asarray(z)))

    def gradient(self, z):
        return np.asarray(objective_grad(jp.asarray(z)))

    def constraints(self, z):
        return np.asarray(constraints(jp.asarray(z)))

    def jacobian(self, z):
        return np.asarray(
            constraint_jac(jp.asarray(z)).reshape(-1)
        )

lb = -1e20 * np.ones(nz)
ub =  1e20 * np.ones(nz)
for i in range(nx):
    lb[i] = x0[i]
    ub[i] = x0[i]

offset = (N + 1) * nx
for k in range(N):
    for j in range(nu):
        idx = offset + k * nu + j
        lb[idx] = -1.0
        ub[idx] = 1.0

cl = np.zeros(N * nx)
cu = np.zeros(N * nx)

In [ ]:
problem = cyipopt.Problem(
    n=nz,
    m=N * nx,
    problem_obj=NLP(),
    lb=lb,
    ub=ub,
    cl=cl,
    cu=cu,
)

problem.add_option("tol", 1e-6)
problem.add_option("print_level", 5)
problem.add_option("max_iter", 500)

xs0 = np.linspace(x0, x_goal, N + 1)
us0 = np.zeros((N, nu))

z0 = np.concatenate([xs0.ravel(), us0.ravel()])

z_opt, info = problem.solve(z0)

xs_opt, us_opt = unpack(z_opt)

In [ ]:
server = viser.ViserServer()
server.scene.add_grid("grid")
scene = ViserMujocoScene(server, model, num_envs=1)
scene.camera_tracking_enabled = False

data = mujoco.MjData(model)
mujoco.mj_kinematics(model, data)
scene.update_from_mjdata(data)

In [ ]:
mjx_model = mjx.put_model(model, contact_approximation="mujoco")
mjx_data = mjx.make_data(mjx_model)

@jax.jit
def step(x, u):
    d = mjx_data.replace(qpos=x[:nq], qvel=x[nq:], ctrl=u)
    d = mjx.step(mjx_model, d)
    return jp.concatenate((d.qpos, d.qvel))

xs_traj = jp.zeros_like(xs_opt)
xs_traj = xs_traj.at[0].set(x0)
for k in range(N):
    tic = time.time()
    x_kp1 = step(xs_traj[k], jp.array(us_opt[k]))
    xs_traj = xs_traj.at[k+1].set(x_kp1)

    data.qpos = np.array(x_kp1[:nq])
    mujoco.mj_kinematics(model, data)
    scene.update_from_mjdata(data)

    remain = model.opt.timestep - (time.time() - tic)
    if remain > 0:
        time.sleep(remain)

ts = np.arange(N + 1) * model.opt.timestep
plt.plot(ts, xs_opt[:, 0], 'r--', label='pusher reference')
plt.plot(ts, xs_opt[:, 1], 'b--', label='object reference')
plt.plot(ts, xs_traj[:, 0], 'r-', label='pusher rollout')
plt.plot(ts, xs_traj[:, 1], 'b-', label='object rollout')
plt.axhline(x_goal[1], color='tab:blue', linestyle=':', label='object goal')
plt.xlabel('Time')
plt.ylabel('Position')
plt.xlim([min(ts), max(ts)])
plt.legend()
plt.show()